# STA 554 HW 9
Jillian Greene

This dataset (CH4_ML_dataset_aqua.csv) is the data I used in my master's thesis to develop a model predicting lake methane emissions based on remotely sensed environmental parameters. See Greene et al. (2026) *Limnology & Oceanography Letters* if you're interested to know more!

In [6]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 13:46:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
ch4_data = pd.read_csv("CH4_ML_dataset_1d_aqua.csv")
ch4_data.head()

,Date,Latitude,Longitude,Site,Temp,rh,CH4_flux,Chla.x,pH,CI,...,TSM,pr,pr3,pr5,pr7,tmmx,vs,Lake,water_temp_K,sim_water
0,2024-05-04,43.223700,-86.28820,Muskegon - Channel,14.7,NaN,0.397998,0.70,8.5,0.0,...,0.527509,7.500459,16.740398,NaN,NaN,298.007956,4.100000,Muskegon,290.16881,290.16881
1,2024-05-04,43.223700,-86.28820,Muskegon - Channel,14.7,NaN,-0.586272,0.70,8.5,0.0,...,0.527509,7.500459,16.740398,NaN,NaN,298.007956,4.100000,Muskegon,290.16881,290.16881
2,2024-05-04,43.223700,-86.28820,Muskegon - Channel,14.7,NaN,0.965204,0.70,8.5,0.0,...,0.527509,7.500459,16.740398,NaN,NaN,298.007956,4.100000,Muskegon,290.16881,290.16881
3,2024-05-04,43.235767,-86.26263,Muskegon - Deep,15.1,NaN,2.625132,0.97,8.4,0.0,...,0.401975,7.120061,15.321439,NaN,NaN,298.042804,4.114472,Muskegon,290.16881,290.16881
4,2024-05-04,43.235767,-86.26263,Muskegon - Deep,15.1,NaN,2.133823,0.97,8.4,0.0,...,0.401975,7.120061,15.321439,NaN,NaN,298.042804,4.114472,Muskegon,290.16881,290.16881


In [8]:
ch4_data = spark.createDataFrame(ch4_data)
ch4_data.show(5)

26/04/09 13:46:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+---------+---------+------------------+----+---+------------+------+---+---+-----------+-----------+------------+--------------+--------------+--------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+---------------+-----------+-----------+----------------+----------------+---+---+----------------+----------------+--------+------------+---------+
|      Date| Latitude|Longitude|              Site|Temp| rh|    CH4_flux|Chla.x| pH| CI|     Chla.y|        PAR|       Kd490|         rtoa1|         rtoa2|         rtoa3|          rtoa4|          rtoa5|          rtoa6|          rtoa7|          rtoa8|          rtoa9|         rtoa10|         rtoa11|         rtoa12|     ADG443|        TSM|              pr|             pr3|pr5|pr7|            tmmx|              vs|    Lake|water_temp_K|sim_water|
+----------+---------+---------+------------------+----+---+------------+------+---+---+-----------+------

In [47]:
from pyspark.ml.feature import SQLTransformer, VectorAssembler

# Set up transformation objects

# 1: Standardize CH4_Flux (log transform) but avoiding NaNs with negatives
sql_logtrans = SQLTransformer(
    statement="""
        SELECT *,
        LOG(1 + CASE 
                    WHEN CH4_Flux < 0 THEN 0
                    ELSE CH4_Flux
                END) AS CH4_Flux_log
        FROM __THIS__
    """
)

# 2: Select desired cols and set CH4_flux_log to 'label'
sql_select = SQLTransformer(
    statement = """
                SELECT Site, CI, `Chla.y` as Chla, PAR, Kd490, ADG443, TSM, pr, sim_water as Temp, CH4_Flux_log as label FROM __THIS__
                """
)

# 3: Convert 'Site' to binary based on site type (Vial vs Chamber)
sql_site_binary = SQLTransformer(
    statement = """
        SELECT *,
        CASE 
            WHEN Site RLIKE '^[0-9]+$' THEN 1.0
            ELSE 0.0
        END AS Site_binary
        FROM __THIS__
    """
)

# 4: Convert Temp from K to C
sql_temp_convert = SQLTransformer(
    statement = """
        SELECT *,
        (Temp - 273.15) AS Temp_C
        FROM __THIS__
    """
)

# Assembler
assembler = VectorAssembler(inputCols = ["Site_binary", "CI", "Chla", "PAR", "Kd490", "ADG443", "TSM", "pr", "Temp"],
                            outputCol = "features", handleInvalid = 'keep')

In [48]:
# Set up the train/test split 
train, test = ch4_data.randomSplit([0.8,0.2], seed = 120301)

In [49]:
# Pipelines
from pyspark.ml import Pipeline
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import ParamGridBuilder

# 1. Linear Regression
lr = LinearRegression()

paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.01, 0.04, 0.06, 0.1]) \
    .addGrid(lr.elasticNetParam, [0, 0.5, 0.8, 0.9, 1]) \
    .build()

lr_pipeline = Pipeline(stages = [sql_logtrans, sql_select, sql_site_binary, sql_temp_convert, assembler, lr])

In [50]:
# Set up crossval object
from pyspark.ml.tuning import CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator

lr_crossval = CrossValidator(estimator = lr_pipeline,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName = "mae"),
                          numFolds=5)

In [51]:
# Run cross-validation on train

# LR
lr_cvModel = lr_crossval.fit(train)


26/04/09 14:05:07 WARN CacheManager: Asked to cache already cached data.
26/04/09 14:05:07 WARN CacheManager: Asked to cache already cached data.
26/04/09 14:05:07 WARN Instrumentation: [2f7bd09e] regParam is zero, which might cause numerical instability and overfitting.
26/04/09 14:05:08 WARN Instrumentation: [9d62bde0] regParam is zero, which might cause numerical instability and overfitting.
26/04/09 14:05:08 WARN Instrumentation: [2c69bf6b] regParam is zero, which might cause numerical instability and overfitting.
26/04/09 14:05:09 WARN Instrumentation: [02aef228] regParam is zero, which might cause numerical instability and overfitting.
26/04/09 14:05:09 WARN Instrumentation: [f4d8a1f6] regParam is zero, which might cause numerical instability and overfitting.
26/04/09 14:05:19 WARN Instrumentation: [a4a4e85a] regParam is zero, which might cause numerical instability and overfitting.
26/04/09 14:05:20 WARN Instrumentation: [0e692a28] regParam is zero, which might cause numerical i

In [52]:
# Crossval on test

# LR
lr_cvModel.transform(test).select("features", "label", "prediction").show(5)
lr_test_error = RegressionEvaluator(metricName = "mae").evaluate(lr_cvModel.transform(test))
print("LR:", lr_test_error)

+--------------------+------------------+------------------+
|            features|             label|        prediction|
+--------------------+------------------+------------------+
|[0.0,0.0,0.988188...|0.6755960638504697|1.4705262470889284|
|[0.0,0.0,0.301181...|4.4051692302668695| 2.368621903622193|
|[0.0,0.0,0.301181...| 4.398937362151815| 2.368621903622193|
|[0.0,0.0,0.353455...| 4.399274405022227|2.2267646497677607|
|[0.0,0.0,0.974803...| 4.044275503590655|1.8062982828025458|
+--------------------+------------------+------------------+
only showing top 5 rows
LR: 1.041753409620483
